<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/extras/modelos_adicionales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EXTRA · Modelos adicionales (fuera de la consigna)

> **Este notebook no forma parte de lo que pide la consigna.** Es un agregado para comparar los modelos
> obligatorios (Keras y YOLO `n`) con otros enfoques: DINOv2, CLIP, ConvNeXt-Tiny / EfficientNetV2-S y
> YOLO11s como clasificador. Usa el mismo split compartido y no modifica nada de lo común.

**Dónde correrlo**
- **Colab (recomendado):** con el botón de arriba, o desde el README. *Entorno de ejecución → Cambiar tipo de
  entorno → T4*. El setup clona el repo, descarga el dataset y monta Drive (`drive.mount`).
  Los resultados finales quedan con el formato común del equipo, una carpeta por modelo y modo
  (`Mi unidad/PDI-TP/resultados/<modelo>/<modo>/`). Las features y las predicciones por fold quedan en
  `Mi unidad/PDI-TP/cache/modelos_extra/`: si se corta la sesión, al volver a correr se saltea lo ya hecho.
- **Local:** con el `.venv` del repo más `torch`, `torchvision`, `transformers` y `ultralytics`. DINOv2 y CLIP
  andan bien en CPU o en Apple Silicon; el fine-tuning y los detectores son muy lentos sin GPU NVIDIA.

**Protocolo**
- Todo se ajusta y se elige **solo con train** (738 imágenes), con validación cruzada de 5 folds
  **agrupada por zona espacial** y estratificada por fuente + con/sin paneles (mismo criterio que el split).
- El `val` compartido del equipo (162 imágenes) se usa acá como **test**: se evalúa **una sola vez al final**
  (sección *Evaluación final*), con los modelos ya elegidos.
- Cada modelo se registra con `registrar(...)`: sus probabilidades out-of-fold en train (para el comparador)
  y una función que lo reentrena con todo train, guarda sus pesos y predice test (solo se llama al final).
- En la evaluación final cada modelo se guarda con `resultados.guardar_corrida(...)`: pesos, métricas y
  predicción contra el real de cada imagen de val, igual que los modelos de la consigna.

In [ ]:
# --- Setup (local o Colab) ---
import os
import random
import subprocess
import sys
from pathlib import Path

INTEGRANTE = os.environ.get("PDI_INTEGRANTE", "nombre")  # reemplazar por tu nombre
EN_COLAB = "google.colab" in sys.modules
USAR_DRIVE = EN_COLAB  # en Colab, resultados y caché en la carpeta compartida de Drive (ver README)

if EN_COLAB:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    # No se instala requirements.txt: fija numpy/pandas para la exploración y puede chocar con el torch de Colab.
    # Colab ya trae torch, torchvision, transformers y scikit-learn; falta Ultralytics (detectores).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
else:
    repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdi_tp" / "entorno.py").is_file())
os.chdir(repo)
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from pdi_tp import datos, entorno, resultados, split

rutas = entorno.preparar(INTEGRANTE, usar_drive=USAR_DRIVE)
# Resultados finales: resultados/<modelo>/<modo>/ con resultados.guardar_corrida (ver Evaluación final).
SALIDA = rutas.analisis / "modelos_extra"  # tablas de la validación cruzada en train
CACHE = rutas.cache / "modelos_extra"  # features y predicciones por fold (se reutilizan si se corta)
for carpeta in (SALIDA, CACHE):
    carpeta.mkdir(parents=True, exist_ok=True)
# Archivos de trabajo de los detectores (muchos archivos chicos): disco local, no Drive
TRABAJO = Path("/content/trabajo") if EN_COLAB else CACHE / "trabajo"

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
pd.set_option("display.precision", 3)
plt.rcParams["figure.dpi"] = 80
print(f"Dispositivo: {DEVICE}   torch {torch.__version__}   Colab: {EN_COLAB}")

## Datos: train y test

In [ ]:
tabla = split.tabla_imagenes(rutas).set_index("archivo")
conjuntos = split.cargar_split(rutas)
train = tabla.loc[[p.name for p in conjuntos["train"]]].reset_index()
test = tabla.loc[[p.name for p in conjuntos["val"]]].reset_index()  # el val compartido = nuestro test
y_train = train["con_paneles"].astype(int).to_numpy()
y_test = test["con_paneles"].astype(int).to_numpy()
todos = list(train["archivo"]) + list(test["archivo"])  # para extraer features una sola vez
n_train = len(train)

print(f"train: {len(train)} imágenes ({(y_train == 0).sum()} sin paneles)   "
      f"test: {len(test)} imágenes ({(y_test == 0).sum()} sin paneles)")

## Validación cruzada y registro de modelos

Métricas: la clase positiva es *con paneles* (85 %), así que además de accuracy/F1 se miran
`recall_sin_paneles` (cuántas imágenes sin paneles detecta), `f1_macro` y `balanced_accuracy`, que no
se inflan por el desbalance. `roc_auc` y `avg_precision` no dependen del umbral.

In [ ]:
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold

N_FOLDS = 5
estrato = train["fuente"] + "_" + train["con_paneles"].astype(str)
FOLDS = list(StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEMILLA)
             .split(train, estrato, groups=train["grupo"]))


def metricas(y, proba, umbral=0.5):
    pred = (proba >= umbral).astype(int)
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred),
        "f1": f1_score(y, pred),
        "recall_sin_paneles": recall_score(y, pred, pos_label=0),
        "f1_macro": f1_score(y, pred, average="macro"),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "roc_auc": roc_auc_score(y, proba),
        "avg_precision": average_precision_score(y, proba),
    }


def oof_proba(X, crear_modelo):
    """Probabilidades out-of-fold en train: cada imagen la predice un modelo que no la vio a ella ni a sus vecinas."""
    proba = np.zeros(len(X))
    for idx_tr, idx_va in FOLDS:
        modelo = crear_modelo().fit(X[idx_tr], y_train[idx_tr])
        proba[idx_va] = modelo.predict_proba(X[idx_va])[:, 1]
    return proba


REGISTRO = {}  # nombre -> familia, params, oof (train), predecir_test y carpeta <modelo>/<modo>


def registrar(nombre, familia, params, oof, predecir_test, modelo, modo, sin_pesos=False):
    """`predecir_test(carpeta)` reentrena con todo train, guarda los pesos en `carpeta` y devuelve
    p(con paneles) para test. Solo se llama en la evaluación final."""
    REGISTRO[nombre] = {"familia": familia, "params": params, "oof": oof, "predecir_test": predecir_test,
                        "modelo": modelo, "modo": modo, "sin_pesos": sin_pesos}
    m = metricas(y_train, oof)
    print(f"Registrado {nombre}: f1_macro={m['f1_macro']:.3f}  recall_sin_paneles={m['recall_sin_paneles']:.3f}  "
          f"roc_auc={m['roc_auc']:.3f}")


def errores_por_fuente(y, proba, fuentes, umbral=0.5):
    pred = (proba >= umbral).astype(int)
    df = pd.DataFrame({"fuente": fuentes, "y": y, "pred": pred})
    return df.groupby("fuente").apply(lambda g: pd.Series({
        "imagenes": len(g),
        "sin_paneles": (g["y"] == 0).sum(),
        "accuracy": (g["y"] == g["pred"]).mean(),
        "falsos_positivos": ((g["y"] == 0) & (g["pred"] == 1)).sum(),
        "falsos_negativos": ((g["y"] == 1) & (g["pred"] == 0)).sum(),
    }), include_groups=False)


print(f"{N_FOLDS} folds; tamaño de cada fold de validación: {[len(v) for _, v in FOLDS]}")

## DINOv2 + clasificador lineal

**Qué es.** DINOv2 (Meta, 2023) es un Vision Transformer entrenado **sin etiquetas** (autosupervisado) sobre
142 M de imágenes. Sus features son lo bastante generales como para que un clasificador lineal encima
funcione muy bien con pocos datos: con 738 imágenes no hace falta (ni conviene) reentrenar la red.

**Cómo se usa acá**
- La red queda **congelada**: se extraen features una vez y se cachean en disco.
- Entrada: la imagen completa redimensionada a 224×224 (múltiplo del parche de 14 px → 16×16 parches).
  No se usa el recorte central del preprocesado por defecto, porque cortaría paneles en los bordes.
- Features: el token `CLS` y el promedio de los tokens de parche (`media`).
- TTA opcional: promedio de las features de las 4 rotaciones de 90° (en imágenes aéreas no hay "arriba").
- Clasificador: regresión logística con `class_weight="balanced"` (compensa el 85/15).

**Qué se compara con CV en train:** tamaño de la red (`small` 21 M parámetros, `base` 86 M), tipo de
features, TTA y regularización `C`.

In [ ]:
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModel

MEDIA_IMAGENET = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
DESVIO_IMAGENET = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def cargar_lote(archivos, lado):
    imgs = [np.asarray(Image.open(rutas.imagenes / a).convert("RGB").resize((lado, lado), Image.BICUBIC))
            for a in archivos]
    x = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float() / 255
    return (x - MEDIA_IMAGENET) / DESVIO_IMAGENET


@torch.no_grad()
def features_dinov2(variante, archivos, lado=224, tta=False, lote=32):
    """Devuelve (cls, media) para cada archivo. Cachea en disco."""
    cache = CACHE / f"dinov2-{variante}_{lado}_{'tta' if tta else 'sin-tta'}.npz"
    if cache.exists():
        z = np.load(cache)
        if list(z["archivos"]) == list(archivos):
            return z["cls"], z["media"]
    modelo = AutoModel.from_pretrained(f"facebook/dinov2-{variante}").to(DEVICE).eval()
    cls, media = [], []
    for i in tqdm(range(0, len(archivos), lote), desc=f"dinov2-{variante} tta={tta}"):
        x = cargar_lote(archivos[i:i + lote], lado)
        vistas = [torch.rot90(x, k, dims=(2, 3)) for k in range(4)] if tta else [x]
        c = m = 0
        for v in vistas:
            h = modelo(pixel_values=v.to(DEVICE)).last_hidden_state.float().cpu()
            c, m = c + h[:, 0], m + h[:, 1:].mean(1)
        cls.append((c / len(vistas)).numpy())
        media.append((m / len(vistas)).numpy())
    cls, media = np.concatenate(cls), np.concatenate(media)
    np.savez(cache, archivos=np.array(archivos), cls=cls, media=media)
    del modelo
    if DEVICE == "mps":
        torch.mps.empty_cache()
    return cls, media

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def regresion_logistica(C):
    return lambda: make_pipeline(StandardScaler(),
                                 LogisticRegression(C=C, class_weight="balanced", max_iter=5000))


VARIANTES, TTAS, TIPOS, CS = ["small", "base"], [False, True], ["cls", "cls+media"], [0.001, 0.01, 0.1, 1.0]

# 1) Features (lo lento: cada combinación variante/TTA tiene su propia barra; después queda en caché)
features = {}
for variante in VARIANTES:
    for tta in TTAS:
        cls, media = features_dinov2(variante, todos, tta=tta)
        features[variante, tta] = {"cls": cls, "cls+media": np.hstack([cls, media])}

# 2) Grilla de validación cruzada (5 folds por configuración)
filas, candidatos = [], {}
configs = [(v, t, f, C) for v in VARIANTES for t in TTAS for f in TIPOS for C in CS]
for variante, tta, tipo, C in tqdm(configs, desc="CV grilla"):
    F = features[variante, tta][tipo]
    oof = oof_proba(F[:n_train], regresion_logistica(C))
    params = {"variante": variante, "tta": tta, "features": tipo, "C": C}
    candidatos[tuple(params.values())] = (params, F, oof)
    filas.append({**params, "dim": F.shape[1], **metricas(y_train, oof)})

grilla = pd.DataFrame(filas).sort_values("f1_macro", ascending=False)
grilla.to_csv(SALIDA / "dinov2_grilla_cv.csv", index=False)
grilla.head(12)

Efecto de cada decisión (promedio de `f1_macro` y `roc_auc` sobre el resto de la grilla):

In [ ]:
for col in ["variante", "tta", "features", "C"]:
    display(grilla.groupby(col)[["f1_macro", "recall_sin_paneles", "roc_auc"]].mean())

Se registra la mejor configuración de cada tamaño (elegida **solo** con la CV de train):

In [ ]:
for variante in ["small", "base"]:
    mejor = grilla[grilla["variante"] == variante].iloc[0]
    params, F, oof = candidatos[(mejor["variante"], mejor["tta"], mejor["features"], mejor["C"])]

    def predecir_test(carpeta, F=F, C=params["C"]):
        modelo = regresion_logistica(C)().fit(F[:n_train], y_train)
        joblib.dump(modelo, carpeta / "modelo.joblib")
        return modelo.predict_proba(F[n_train:])[:, 1]

    registrar(f"DINOv2-{variante} + LR", "DINOv2 (lineal)", params, oof, predecir_test,
              modelo=f"dinov2-{variante}", modo="linear-probe")

### Análisis de errores (out-of-fold, train)

In [ ]:
nombre = max(REGISTRO, key=lambda n: metricas(y_train, REGISTRO[n]["oof"])["f1_macro"])
oof = REGISTRO[nombre]["oof"]
print(nombre)
display(errores_por_fuente(y_train, oof, train["fuente"]))

fig, ax = plt.subplots(figsize=(4, 4))
cm = confusion_matrix(y_train, oof >= 0.5)
ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, v, ha="center", va="center", fontsize=14)
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["sin", "con"], yticklabels=["sin", "con"],
       xlabel="predicho", ylabel="real", title="Matriz de confusión (OOF)")
plt.tight_layout()

In [ ]:
# Los errores más confiados: falsos positivos (dice "con paneles" y no hay) y falsos negativos
errores = train.assign(proba=oof)
fp = errores[(errores["con_paneles"] == False) & (errores["proba"] >= 0.5)].nlargest(6, "proba")
fn = errores[(errores["con_paneles"] == True) & (errores["proba"] < 0.5)].nsmallest(6, "proba")
fig, axs = plt.subplots(2, 6, figsize=(18, 6.5))
for ax in axs.flat:
    ax.set_axis_off()
for fila, (titulo, df) in zip(axs, [("FP", fp), ("FN", fn)]):
    for ax, (_, r) in zip(fila, df.iterrows()):
        datos.dibujar_cajas(ax, rutas, rutas.imagenes / r["archivo"])
        ax.set_title(f"{titulo} p={r['proba']:.2f}\n{r['fuente']}", fontsize=9)
plt.tight_layout()

### ¿Qué separan las features? (PCA)

Si los puntos se agrupan más por **fuente** que por **con/sin paneles**, el clasificador podría estar
aprendiendo a reconocer la fuente (el atajo que vimos en la exploración).

In [ ]:
from sklearn.decomposition import PCA

params = REGISTRO[nombre]["params"]
F = candidatos[tuple(params.values())][1][:n_train]
Z = PCA(2, random_state=SEMILLA).fit_transform(StandardScaler().fit_transform(F))
fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
for etiqueta, color in [(True, "tab:orange"), (False, "tab:gray")]:
    sel = train["con_paneles"] == etiqueta
    axs[0].scatter(Z[sel, 0], Z[sel, 1], s=8, c=color, alpha=0.6, label="con paneles" if etiqueta else "sin paneles")
for f in sorted(train["fuente"].unique()):
    sel = train["fuente"] == f
    axs[1].scatter(Z[sel, 0], Z[sel, 1], s=8, alpha=0.6, label=f)
axs[0].set_title("PCA de las features: con/sin paneles")
axs[1].set_title("PCA de las features: fuente")
for ax in axs:
    ax.legend(markerscale=3)
plt.tight_layout()

## CLIP zero-shot

**Qué es.** CLIP (OpenAI, 2021) entrena un codificador de imágenes y uno de texto para que una imagen y su
descripción queden cerca en el mismo espacio. Con eso se puede clasificar **sin entrenar**: se escriben
descripciones de cada clase (*prompts*) y se elige la más parecida a la imagen.

**Cómo se usa acá**
- Para cada clase se promedian los embeddings de varios prompts (*prompt ensemble*), lo que suele ser más
  robusto que un solo texto.
- p(con paneles) = softmax de la similitud coseno (escalada por la temperatura de CLIP) contra las dos clases.
- Como no hay entrenamiento, la predicción "out-of-fold" en train es simplemente la predicción directa.
- Se comparan dos tamaños (`ViT-B/32`, 151 M parámetros; `ViT-L/14`, 428 M), TTA de rotaciones y dos
  juegos de prompts.

Además se registran dos variantes que **sí** usan train, para medir cuánto aporta entrenar:
- **Zero-shot + umbral**: lo único que se aprende es el umbral de decisión (CLIP no está calibrado: 0.5 no
  tiene por qué ser el mejor corte). Se elige dentro de cada fold.
- **CLIP + LR**: las mismas features de imagen con una regresión logística, como DINOv2.

*RemoteCLIP* (CLIP ajustado a imágenes satelitales) queda afuera: sus pesos se distribuyen como checkpoint
`.pt` de terceros, que se carga con pickle y puede ejecutar código.

In [ ]:
from functools import lru_cache

from transformers import CLIPModel, CLIPProcessor

MODELOS_CLIP = {"ViT-B/32": "openai/clip-vit-base-patch32", "ViT-L/14": "openai/clip-vit-large-patch14"}

PROMPTS = {
    "simple": {
        1: ["an aerial photo of solar panels"],
        0: ["an aerial photo"],
    },
    "ensamble": {
        1: ["a satellite image of solar panels on a roof",
            "an aerial photo of rooftop solar panels",
            "an aerial view of photovoltaic panels",
            "a satellite photo of a solar farm",
            "an overhead image of buildings with solar panels"],
        0: ["a satellite image of roofs without solar panels",
            "an aerial photo of houses and trees",
            "an aerial view of a residential neighborhood",
            "a satellite photo of streets and buildings",
            "an overhead image of vegetation and roads"],
    },
}


@lru_cache(maxsize=1)  # un modelo a la vez en memoria (8 GB de RAM)
def cargar_clip(modelo_id):
    return CLIPModel.from_pretrained(modelo_id).to(DEVICE).eval(), CLIPProcessor.from_pretrained(modelo_id)


def _tensor(salida):
    """transformers 4 devuelve el embedding directo; transformers 5, un objeto con .pooler_output."""
    return salida if torch.is_tensor(salida) else salida.pooler_output


def _normalizar(x):
    return x / x.norm(dim=-1, keepdim=True)


@torch.no_grad()
def embeddings_imagen_clip(nombre, archivos, tta=False, lote=32):
    cache = CACHE / f"clip-{nombre.replace('/', '')}_{'tta' if tta else 'sin-tta'}.npz"
    if cache.exists():
        z = np.load(cache)
        if list(z["archivos"]) == list(archivos):
            return z["emb"]
    modelo, procesador = cargar_clip(MODELOS_CLIP[nombre])
    embs = []
    for i in tqdm(range(0, len(archivos), lote), desc=f"CLIP {nombre} tta={tta}"):
        imgs = [Image.open(rutas.imagenes / a).convert("RGB") for a in archivos[i:i + lote]]
        x = procesador(images=imgs, return_tensors="pt")["pixel_values"]
        vistas = [torch.rot90(x, k, dims=(2, 3)) for k in range(4)] if tta else [x]
        e = sum(_normalizar(_tensor(modelo.get_image_features(pixel_values=v.to(DEVICE))).float().cpu())
                for v in vistas)
        embs.append(_normalizar(e).numpy())
    emb = np.concatenate(embs)
    np.savez(cache, archivos=np.array(archivos), emb=emb)
    return emb


@torch.no_grad()
def embeddings_texto_clip(nombre, prompts):
    """(2, d): embedding promedio de los prompts de cada clase (fila 0 = sin paneles, fila 1 = con paneles)."""
    modelo, procesador = cargar_clip(MODELOS_CLIP[nombre])
    filas = []
    for clase in (0, 1):
        tok = procesador(text=prompts[clase], return_tensors="pt", padding=True).to(DEVICE)
        e = _normalizar(_tensor(modelo.get_text_features(**tok)).float().cpu())
        filas.append(_normalizar(e.mean(0)))
    return torch.stack(filas).numpy(), modelo.logit_scale.exp().item()


def proba_zero_shot(emb_img, emb_txt, escala):
    logits = escala * emb_img @ emb_txt.T
    logits -= logits.max(1, keepdims=True)
    p = np.exp(logits)
    return p[:, 1] / p.sum(1)

In [ ]:
filas_clip, scores_clip, emb_clip = [], {}, {}
for nombre in MODELOS_CLIP:
    for tta in [False, True]:
        emb_clip[nombre, tta] = embeddings_imagen_clip(nombre, todos, tta=tta)
    for juego, prompts in PROMPTS.items():
        emb_txt, escala = embeddings_texto_clip(nombre, prompts)
        for tta in [False, True]:
            p = proba_zero_shot(emb_clip[nombre, tta], emb_txt, escala)
            params = {"modelo": nombre, "tta": tta, "prompts": juego}
            scores_clip[tuple(params.values())] = (params, p)
            filas_clip.append({**params, **metricas(y_train, p[:n_train])})
    cargar_clip.cache_clear()
    if DEVICE == "mps":
        torch.mps.empty_cache()

grilla_clip = pd.DataFrame(filas_clip).sort_values("roc_auc", ascending=False)
grilla_clip.to_csv(SALIDA / "clip_grilla.csv", index=False)
grilla_clip

Para elegir la configuración zero-shot se usa `roc_auc`, que no depende del umbral: mide si CLIP
**ordena** bien las imágenes. Las métricas con umbral 0.5 pueden ser malas aunque el orden sea bueno.

In [ ]:
class UmbralCalibrado:
    """Lo único que aprende es el umbral sobre el score zero-shot: el que maximiza balanced accuracy.
    predict_proba reescala el score para que ese umbral quede en 0.5 (así se compara con el resto)."""

    def fit(self, X, y):
        s = X.ravel()
        self.umbral_ = max(np.unique(s), key=lambda u: balanced_accuracy_score(y, s >= u))
        return self

    def predict_proba(self, X):
        logit = lambda q: np.log(q / (1 - q))
        s = np.clip(X.ravel(), 1e-6, 1 - 1e-6)
        u = np.clip(self.umbral_, 1e-6, 1 - 1e-6)
        p = 1 / (1 + np.exp(-(logit(s) - logit(u))))
        return np.column_stack([1 - p, p])


mejor = grilla_clip.iloc[0]
params, p = scores_clip[(mejor["modelo"], mejor["tta"], mejor["prompts"])]
S = p.reshape(-1, 1)

CLIP_SLUG = "clip-" + params["modelo"].lower().replace("/", "")  # p. ej. clip-vit-l14

# Zero-shot: no hay pesos entrenados (se usan los de OpenAI); los prompts quedan en params.
registrar(f"CLIP {params['modelo']} zero-shot", "CLIP zero-shot", {**params, "prompts_texto": PROMPTS[params["prompts"]]},
          p[:n_train], lambda carpeta, p=p: p[n_train:], modelo=CLIP_SLUG, modo="zero-shot", sin_pesos=True)

oof = oof_proba(S[:n_train], UmbralCalibrado)
registrar(f"CLIP {params['modelo']} zero-shot + umbral", "CLIP zero-shot", {**params, "umbral": "CV"},
          oof, lambda carpeta, S=S: UmbralCalibrado().fit(S[:n_train], y_train).predict_proba(S[n_train:])[:, 1],
          modelo=CLIP_SLUG, modo="zero-shot-umbral", sin_pesos=True)
print(f"Umbral elegido con todo train: {UmbralCalibrado().fit(S[:n_train], y_train).umbral_:.3f}")

# Linear probe con las mismas features de imagen (mismo modelo y TTA que el mejor zero-shot)
F = emb_clip[params["modelo"], params["tta"]]
filas_lp = [{"C": C, **metricas(y_train, oof_proba(F[:n_train], regresion_logistica(C)))}
            for C in [0.001, 0.01, 0.1, 1.0]]
C = pd.DataFrame(filas_lp).sort_values("f1_macro", ascending=False).iloc[0]["C"]
def predecir_test_clip_lr(carpeta, F=F, C=C):
    modelo = regresion_logistica(C)().fit(F[:n_train], y_train)
    joblib.dump(modelo, carpeta / "modelo.joblib")
    return modelo.predict_proba(F[n_train:])[:, 1]


registrar(f"CLIP {params['modelo']} + LR", "CLIP (lineal)", {**params, "C": C},
          oof_proba(F[:n_train], regresion_logistica(C)), predecir_test_clip_lr,
          modelo=CLIP_SLUG, modo="linear-probe")

In [ ]:
# ¿Dónde se equivoca el zero-shot? Distribución del score por clase y por fuente
p_train = scores_clip[(mejor["modelo"], mejor["tta"], mejor["prompts"])][1][:n_train]
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for etiqueta, color in [(1, "tab:orange"), (0, "tab:gray")]:
    axs[0].hist(p_train[y_train == etiqueta], bins=40, alpha=0.6, color=color,
                label="con paneles" if etiqueta else "sin paneles")
axs[0].axvline(0.5, color="k", ls="--", lw=1)
axs[0].set(title="Score zero-shot en train", xlabel="p(con paneles)")
axs[0].legend()
train.assign(score=p_train).boxplot(column="score", by=["fuente", "con_paneles"], ax=axs[1], rot=45)
axs[1].set(title="Score por fuente y etiqueta", xlabel="")
plt.suptitle("")
plt.tight_layout()
display(errores_por_fuente(y_train, REGISTRO[f"CLIP {params['modelo']} zero-shot + umbral"]["oof"], train["fuente"]))

## Fine-tuning: ConvNeXt-Tiny y EfficientNetV2-S

**Qué son.** Dos CNN modernas preentrenadas en ImageNet (torchvision), más recientes que las de la consigna:
- **ConvNeXt-Tiny** (2022, 28 M parámetros): una ResNet "modernizada" con decisiones de diseño tomadas de los
  Vision Transformers (kernels de 7×7, LayerNorm, GELU, menos activaciones).
- **EfficientNetV2-S** (2021, 21 M): sucesora de EfficientNetB0, con bloques *Fused-MBConv* en las primeras
  etapas y escalado progresivo; entrena más rápido que la V1.

**A diferencia de DINOv2 y CLIP, acá se entrena toda la red** (fine-tuning completo):
- Entrada 256×256 (el tamaño original: las CNN no necesitan 224).
- Augmentation para imágenes aéreas: flips, rotaciones de 90° y un poco de color.
- AdamW con *learning rate* diferencial: 1e-3 la cabeza nueva, 1e-4 el extractor preentrenado (para no
  destruir lo aprendido en ImageNet), con *one-cycle* (calentamiento y bajada).
- Pérdida con pesos de clase (desbalance 85/15) y *label smoothing* (las etiquetas tienen algo de ruido).
- Predicción con TTA de 4 rotaciones, igual que DINOv2 y CLIP.
- Se entrena una red por fold (5 por arquitectura). Las predicciones de cada fold se guardan en el caché:
  si se corta Colab, al volver a correr se retoma desde el fold que faltaba.

Tiempo aproximado en una T4: unos minutos por fold y por red.

In [ ]:
import hashlib
import json
import shutil
import time

import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2 as T

FT_ARQUITECTURAS = {
    "ConvNeXt-Tiny": (torchvision.models.convnext_tiny, torchvision.models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1),
    "EfficientNetV2-S": (torchvision.models.efficientnet_v2_s, torchvision.models.EfficientNet_V2_S_Weights.IMAGENET1K_V1),
}
FT = {"epocas": 15, "lote": 32, "lr_cabeza": 1e-3, "lr_base": 1e-4, "weight_decay": 0.05, "label_smoothing": 0.05}
TRABAJADORES = 2 if EN_COLAB else 0  # en macOS los workers de DataLoader no andan bien dentro de notebooks


def firma(config):
    """Identifica una configuración en los nombres del caché: si se cambian hiperparámetros, se recalcula."""
    return hashlib.md5(json.dumps(config, sort_keys=True).encode()).hexdigest()[:8]


if DEVICE != "cuda":
    print("Aviso: sin GPU NVIDIA el fine-tuning es muy lento. Pensado para Colab con T4.")

# Todas las imágenes en memoria como uint8 (900 × 3 × 256 × 256 ≈ 180 MB)
IMGS = {a: torch.from_numpy(datos.leer_imagen(rutas.imagenes / a).copy()).permute(2, 0, 1) for a in todos}

AUMENTAR = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomChoice([T.RandomRotation((k, k)) for k in (0, 90, 180, 270)]),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
])
NORMALIZAR = T.Compose([T.ToDtype(torch.float32, scale=True),
                        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


class Imagenes(Dataset):
    def __init__(self, archivos, y=None, aumentar=False):
        self.archivos, self.y, self.aumentar = list(archivos), y, aumentar

    def __len__(self):
        return len(self.archivos)

    def __getitem__(self, i):
        x = IMGS[self.archivos[i]]
        x = NORMALIZAR(AUMENTAR(x) if self.aumentar else x)
        return x if self.y is None else (x, int(self.y[i]))


def crear_red(nombre):
    constructor, pesos = FT_ARQUITECTURAS[nombre]
    red = constructor(weights=pesos)
    # Se reemplaza la última capa lineal (1000 clases de ImageNet) por una de 2 clases
    ultima = max(i for i, m in enumerate(red.classifier) if isinstance(m, nn.Linear))
    red.classifier[ultima] = nn.Linear(red.classifier[ultima].in_features, 2)
    return red


def entrenar_red(nombre, archivos, y):
    torch.manual_seed(SEMILLA)
    red = crear_red(nombre).to(DEVICE)
    cabeza = list(red.classifier.parameters())
    en_cabeza = {id(p) for p in cabeza}
    base = [p for p in red.parameters() if id(p) not in en_cabeza]
    opt = torch.optim.AdamW([{"params": base, "lr": FT["lr_base"]}, {"params": cabeza, "lr": FT["lr_cabeza"]}],
                            weight_decay=FT["weight_decay"])
    carga = DataLoader(Imagenes(archivos, y, aumentar=True), batch_size=FT["lote"], shuffle=True,
                       num_workers=TRABAJADORES, drop_last=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[FT["lr_base"], FT["lr_cabeza"]],
                                                total_steps=FT["epocas"] * len(carga), pct_start=0.1)
    pesos_clase = torch.tensor(len(y) / (2 * np.bincount(y, minlength=2)), dtype=torch.float32, device=DEVICE)
    perdida = nn.CrossEntropyLoss(weight=pesos_clase, label_smoothing=FT["label_smoothing"])
    amp = DEVICE == "cuda"  # precisión mixta: ~2x más rápido en la T4
    escalador = torch.amp.GradScaler("cuda", enabled=amp)

    red.train()
    for _ in tqdm(range(FT["epocas"]), desc=f"{nombre} épocas", leave=False):
        for x, t in carga:
            x, t = x.to(DEVICE, non_blocking=True), t.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                loss = perdida(red(x), t)
            opt.zero_grad(set_to_none=True)
            escalador.scale(loss).backward()
            escalador.step(opt)
            escalador.update()
            sched.step()
    return red.eval()


@torch.no_grad()
def predecir_red(red, archivos, tta=True):
    carga = DataLoader(Imagenes(archivos), batch_size=64, num_workers=TRABAJADORES)
    probas = []
    for x in carga:
        x = x.to(DEVICE)
        vistas = [torch.rot90(x, k, dims=(2, 3)) for k in range(4)] if tta else [x]
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            p = sum(red(v).float().softmax(1)[:, 1] for v in vistas) / len(vistas)
        probas.append(p.cpu().numpy())
    return np.concatenate(probas)


def liberar_gpu():
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


def oof_red(nombre):
    proba = np.zeros(n_train)
    for k, (idx_tr, idx_va) in enumerate(FOLDS):
        cache = CACHE / f"ft-{nombre}_{firma(FT)}_fold{k}.npy"
        if cache.exists():
            proba[idx_va] = np.load(cache)
            continue
        inicio = time.time()
        red = entrenar_red(nombre, train["archivo"].iloc[idx_tr], y_train[idx_tr])
        proba[idx_va] = predecir_red(red, train["archivo"].iloc[idx_va])
        np.save(cache, proba[idx_va])
        print(f"{nombre} fold {k + 1}/{N_FOLDS}: {time.time() - inicio:.0f} s")
        del red
        liberar_gpu()
    return proba

In [ ]:
for nombre in tqdm(FT_ARQUITECTURAS, desc="Arquitecturas"):
    oof = oof_red(nombre)

    def predecir_test(carpeta, nombre=nombre):
        red = entrenar_red(nombre, train["archivo"], y_train)
        torch.save(red.state_dict(), carpeta / "modelo.pt")
        p = predecir_red(red, test["archivo"])
        del red
        liberar_gpu()
        return p

    registrar(f"{nombre} fine-tuning", "CNN fine-tuning", {"red": nombre, **FT, "tta": True}, oof, predecir_test,
              modelo=nombre.lower(), modo="fine-tuning")

display(errores_por_fuente(y_train, REGISTRO["ConvNeXt-Tiny fine-tuning"]["oof"], train["fuente"]))

## Detectores como clasificador: YOLO11s (y RT-DETR opcional)

La consigna pide usar el detector como clasificador: **"con paneles" si detecta al menos un panel**. Acá se
hace con detectores más grandes que los `n` obligatorios, para ver si un buen detector le gana a los
clasificadores también en la tarea de clasificación.

**Qué son**
- **YOLO11s**: la versión *small* de YOLO11 (9 M parámetros contra 2.6 M de la `n`). Mismo diseño, más ancho y
  profundo.
- **RT-DETR-L** (opcional, `INCLUIR_RTDETR`): detector basado en transformers (Baidu, 2023). En vez de anclas
  y supresión de no-máximos, predice directamente un conjunto de cajas con atención. Es bastante más lento de
  entrenar.

**Cómo se usa acá**
- Se entrena con las cajas de los folds de train. Las 3 clases se fusionan en "panel" (`single_cls=True`):
  para clasificar solo importa si hay algún panel.
- `imgsz=512`: Ultralytics agranda las imágenes de 256 px, lo que ayuda con las cajas chicas. La consigna
  pide 256 para los modelos obligatorios; esto es un extra.
- Augmentation para imágenes aéreas: flips verticales y horizontales y rotaciones (`degrees`).
- Sin validación durante el entrenamiento y con los pesos de la **última época**. Elegir `best.pt` con el fold
  de validación filtraría información a la CV.
- Score de cada imagen = **confianza máxima** de sus detecciones (0 si no detecta nada). Dos reglas:
  1. **La de la consigna**: al menos una caja con `conf ≥ 0.25` (umbral por defecto de Ultralytics).
  2. **Umbral calibrado** sobre la confianza máxima, elegido dentro de cada fold (igual que con CLIP).

Tiempo aproximado en una T4: varios minutos por fold con YOLO11s; RT-DETR tarda bastante más.

In [ ]:
DET = {"epocas": 60, "imgsz": 512, "lote": 16, "degrees": 90.0, "flipud": 0.5, "fliplr": 0.5}
INCLUIR_RTDETR = False  # True para sumar RT-DETR-L (bastante más lento)
MODELOS_DET = {"YOLO11s": "yolo11s.pt", **({"RT-DETR-L": "rtdetr-l.pt"} if INCLUIR_RTDETR else {})}
CONF_CONSIGNA = 0.25
DISPOSITIVO_DET = 0 if DEVICE == "cuda" else DEVICE


def reescalar(score, umbral):
    """Transforma el score para que `umbral` quede en 0.5 (mismo orden; así todos se cortan en 0.5)."""
    logit = lambda q: np.log(q / (1 - q))
    s = np.clip(score, 1e-6, 1 - 1e-6)
    return 1 / (1 + np.exp(-(logit(s) - logit(np.clip(umbral, 1e-6, 1 - 1e-6)))))


def entrenar_detector(nombre, archivos, etiqueta):
    from ultralytics import RTDETR, YOLO

    carpeta = TRABAJO / etiqueta
    carpeta.mkdir(parents=True, exist_ok=True)
    lista = carpeta / "train.txt"
    lista.write_text("\n".join(str(rutas.imagenes / a) for a in archivos) + "\n")
    # Ultralytics busca cada etiqueta cambiando /images/ por /labels/ en la ruta de la imagen.
    # 'val' es obligatorio en el yaml: se pone el mismo train (solo se usa en la validación final interna).
    (carpeta / "data.yaml").write_text(f'train: "{lista}"\nval: "{lista}"\nnc: 1\nnames: ["panel"]\n')

    clase = RTDETR if nombre.startswith("RT-DETR") else YOLO
    clase(MODELOS_DET[nombre]).train(
        data=str(carpeta / "data.yaml"), epochs=DET["epocas"], imgsz=DET["imgsz"], batch=DET["lote"],
        single_cls=True, val=False, degrees=DET["degrees"], flipud=DET["flipud"], fliplr=DET["fliplr"],
        project=str(carpeta), name="run", exist_ok=True, seed=SEMILLA, device=DISPOSITIVO_DET,
        workers=TRABAJADORES, plots=False, verbose=False,
    )
    return clase(str(carpeta / "run" / "weights" / "last.pt"))


def score_detector(modelo, archivos):
    """Confianza máxima de las detecciones de cada imagen (0 si no hay ninguna)."""
    resultados = modelo.predict([str(rutas.imagenes / a) for a in archivos], imgsz=DET["imgsz"], conf=0.001,
                                device=DISPOSITIVO_DET, stream=True, verbose=False)
    return np.array([float(r.boxes.conf.max()) if len(r.boxes) else 0.0 for r in resultados])


def oof_detector(nombre):
    score = np.zeros(n_train)
    for k, (idx_tr, idx_va) in enumerate(FOLDS):
        cache = CACHE / f"det-{nombre}_{firma(DET)}_fold{k}.npy"
        if cache.exists():
            score[idx_va] = np.load(cache)
            continue
        inicio = time.time()
        modelo = entrenar_detector(nombre, train["archivo"].iloc[idx_tr], f"{nombre}_fold{k}")
        score[idx_va] = score_detector(modelo, train["archivo"].iloc[idx_va])
        np.save(cache, score[idx_va])
        print(f"{nombre} fold {k + 1}/{N_FOLDS}: {time.time() - inicio:.0f} s")
        del modelo
        liberar_gpu()
    return score


SCORES_TEST_DET = {}  # el modelo final se entrena una sola vez y lo usan las dos reglas


def score_test_detector(nombre, carpeta):
    """Entrena (una vez) el detector final con todo train, copia sus pesos a `carpeta` y da el score de test."""
    pesos = TRABAJO / f"{nombre}_final" / "run" / "weights" / "last.pt"
    if nombre not in SCORES_TEST_DET:
        modelo = entrenar_detector(nombre, train["archivo"], f"{nombre}_final")
        SCORES_TEST_DET[nombre] = score_detector(modelo, test["archivo"])
    shutil.copy(pesos, carpeta / "modelo.pt")
    return SCORES_TEST_DET[nombre]

In [ ]:
for nombre in tqdm(MODELOS_DET, desc="Detectores"):
    s = oof_detector(nombre)
    params = {"modelo": nombre, **DET}

    registrar(f"{nombre} ≥1 caja (conf≥{CONF_CONSIGNA})", "Detector como clasificador",
              {**params, "regla": f"conf>={CONF_CONSIGNA}"}, reescalar(s, CONF_CONSIGNA),
              lambda carpeta, nombre=nombre: reescalar(score_test_detector(nombre, carpeta), CONF_CONSIGNA),
              modelo=nombre.lower(), modo="clasificador-conf025")

    S = s.reshape(-1, 1)
    registrar(f"{nombre} + umbral", "Detector como clasificador", {**params, "regla": "umbral CV"},
              oof_proba(S, UmbralCalibrado),
              lambda carpeta, nombre=nombre, S=S: UmbralCalibrado().fit(S, y_train).predict_proba(
                  score_test_detector(nombre, carpeta).reshape(-1, 1))[:, 1],
              modelo=nombre.lower(), modo="clasificador-umbral-cv")
    print(f"{nombre}: umbral elegido con todo train = {UmbralCalibrado().fit(S, y_train).umbral_:.3f}")

display(errores_por_fuente(y_train, REGISTRO[f"YOLO11s ≥1 caja (conf≥{CONF_CONSIGNA})"]["oof"], train["fuente"]))

## Comparador (validación cruzada en train)

In [ ]:
comparacion_cv = pd.DataFrame(
    {nombre: {"familia": r["familia"], **metricas(y_train, r["oof"])} for nombre, r in REGISTRO.items()}
).T.sort_values("f1_macro", ascending=False)
comparacion_cv.to_csv(SALIDA / "comparacion_cv.csv")
comparacion_cv

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for nombre, r in REGISTRO.items():
    fpr, tpr, _ = roc_curve(y_train, r["oof"])
    axs[0].plot(fpr, tpr, label=nombre)
    # Curva PR de la clase minoritaria (sin paneles): la difícil
    p, rc, _ = precision_recall_curve(1 - y_train, 1 - r["oof"])
    axs[1].plot(rc, p, label=nombre)
axs[0].set(title="ROC (OOF train)", xlabel="FPR", ylabel="TPR")
axs[1].set(title="Precisión-recall de 'sin paneles' (OOF train)", xlabel="recall", ylabel="precisión")
for ax in axs:
    ax.legend()
plt.tight_layout()

## Evaluación final en test

**Correr una sola vez, al final**, con los modelos ya elegidos. Si se mira test y después se cambian
modelos o hiperparámetros, test deja de ser una estimación honesta. Por eso está apagado con `EVALUAR_TEST`.
Cada modelo se reentrena con **todo** train y predice las 162 imágenes de test. Cada uno queda guardado
con el formato común (`resultados/<modelo>/<modo>/`: pesos, métricas y predicción contra el real). Los que ya
están completos en Drive se reutilizan; `REPETIR = True` los vuelve a correr (la versión anterior se archiva).

In [ ]:
EVALUAR_TEST = False  # poner True solo para la pasada final
REPETIR = False  # True vuelve a correr modelos que ya tienen resultados completos (archiva los anteriores)

if EVALUAR_TEST:
    probas_test = {}
    for nombre, r in REGISTRO.items():
        carpeta = resultados.ruta_corrida(rutas, r["modelo"], r["modo"])
        if resultados.corrida_completa(carpeta) and not REPETIR:
            guardadas = pd.read_csv(carpeta / "predicciones_val.csv").set_index("archivo")
            probas_test[nombre] = guardadas.loc[test["archivo"], "probabilidad"].to_numpy()
            print(f"Ya guardado, se reutiliza: {r['modelo']}/{r['modo']}")
            continue
        carpeta = resultados.carpeta_corrida(rutas, r["modelo"], r["modo"], reiniciar=REPETIR)
        probas_test[nombre] = r["predecir_test"](carpeta)
        resultados.guardar_corrida(
            rutas, carpeta, test["archivo"], probas_test[nombre], params=r["params"], sin_pesos=r["sin_pesos"],
            metricas_extra={"extra_fuera_de_consigna": True, "familia": r["familia"], "nombre": nombre,
                            "cv_train": metricas(y_train, r["oof"])})

    comparacion_test = pd.DataFrame(
        {nombre: {"familia": REGISTRO[nombre]["familia"], **metricas(y_test, p)} for nombre, p in probas_test.items()}
    ).T.sort_values("f1_macro", ascending=False)
    comparacion_test.to_csv(SALIDA / "comparacion_test.csv")
    display(comparacion_test)

    print("Diferencia test - CV (negativo = rinde peor en test que lo que estimaba la CV):")
    display((comparacion_test.drop(columns="familia") - comparacion_cv.drop(columns="familia")).loc[comparacion_test.index])

    fig, axs = plt.subplots(1, len(probas_test), figsize=(4 * len(probas_test), 4), squeeze=False)
    for ax, (nombre, p) in zip(axs[0], probas_test.items()):
        cm = confusion_matrix(y_test, p >= 0.5)
        ax.imshow(cm, cmap="Blues")
        for (i, j), v in np.ndenumerate(cm):
            ax.text(j, i, v, ha="center", va="center", fontsize=14)
        ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["sin", "con"], yticklabels=["sin", "con"],
               xlabel="predicho", ylabel="real", title=nombre)
    plt.tight_layout()

    for nombre, p in probas_test.items():
        print(nombre)
        display(errores_por_fuente(y_test, p, test["fuente"]))
else:
    print("Evaluación en test desactivada. Poner EVALUAR_TEST = True para la pasada final.")